# Topic 3 — Data Engineering for AI
## Industrial Machine Vision Knowledge Assistant

**Real project data:** Ren et al. (2022) and Hütten et al. (2024), both selected open-access academic PDFs.  
**Selected lab pathway:** **A — PDF/Text only**; images in the papers are not an image-classification dataset.  
**Notebook status:** Executable document pipeline, not a deployed RAG chatbot. No embeddings, LLM, vector database, classification model or answer-quality measurements are performed.

**Lab sequence:** Cell 1 → upload two PDFs → Cell 3 → Cell 4 → Cell 4A → Cell 5A1 → Cell 5A2 → B3 → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9.

Use **Runtime > Run all** only *after* placing the PDFs in the path printed by Cell 1. Run the cells in order. The package `SETUP_README.md` contains the exact upload instructions.

**Source coverage limitation:** These documents describe inspection systems and deep-learning methods; they do not guarantee detailed comparisons of AOI and AXI. A future RAG assistant must abstain when cited evidence is insufficient.


### A1 — The three primary data requirements (from Topic 2)

| Primary requirement | Needed information / representation | Actual sources | Why suitable / constraint |
|---|---|---|---|
| **FR1 — Supported technical explanations** | Industrial machine-vision concepts: optical illumination, sensors, image acquisition, image processing and automated visual inspection | SRC-01 Ren et al. (2022), SRC-02 Hütten et al. (2024) | Two peer-reviewed, text-extractable reviews; content is limited to claims supported by these exact papers |
| **NFR3 — Retrieval relevance** | Coherent, correctly sourced, searchable text passages | Embedded PDF text divided into page-labelled chunks | Compare fixed 500-character chunks with paragraph-/block-aware chunks; **Hit@5 is not tested yet** |
| **RAI2 — Verifiable support** | Source title, DOI, document revision/publication metadata and page numbers | Metadata of both PDFs carried into each output record | Supports *future* citation checks; **metadata by itself does not guarantee that a generated claim is supported** |

**Supporting requirements:** FR3 (source citation), FR4/RAI1 (insufficient evidence), RAI3 (date transparency) and NFR4 (repeatable update) remain important to later RAG implementation and are discussed in `docs/week3_pipeline_decisions.md`; they are not separate primary A1 rows.

**Important project boundary:** Input is *technical literature about visual inspection*, not a dataset of inspection images. **Pathway A (PDF/Text)** is the correct lab pathway. The prepared representation is JSONL containing text and provenance, not an image-classification dataset.


In [ ]:
# CELL 1 — REQUIRED: Google Colab workspace (run first)
import os
from pathlib import Path
BASE_DIR = Path(os.environ.get("WEEK3_BASE_DIR", "/content/week3_project"))
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
for d in (RAW_DIR, PROCESSED_DIR, PIPE_DIR):
    d.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)
print("Pipeline folder:", PIPE_DIR)
print("Next: upload both named PDF files into the Raw folder, then run Cell 3.")

### A2 — Upload the two approved source PDFs (one-time user action)

In Google Colab, use the **Files** sidebar to upload **both** files into `/content/week3_project/data/raw/`:

- `ren_2022_machine_vision_defect_detection.pdf`
- `huetten_2024_deep_learning_visual_inspection.pdf`

These two files are included in the downloaded ZIP under `source_pdfs_local_only/`; do **not** upload them to a public GitHub repository unless the rights/attribution conditions are satisfied. Colab's `/content` directory is temporary. Re-upload after runtime reset.

**Cell 2 (optional practice ZIP)** is intentionally skipped; we use real approved project PDFs rather than synthetic data.

In [ ]:
# CELL 3 — REQUIRED: verify exact input files and inspect both source names
SOURCES = {
    "SRC-01": {
        "filename": "ren_2022_machine_vision_defect_detection.pdf",
        "title": "State of the Art in Defect Detection Based on Machine Vision",
        "authors": "Zhonghe Ren; Fengzhou Fang; Ning Yan; You Wu",
        "publication": "2022 (published online 26 May 2021)",
        "publication_date": "2021-05-26",
        "doi": "https://doi.org/10.1007/s40684-021-00343-6",
        "license": "CC BY 4.0; check third-party figure rights",
        "expected_pages": 31,
    },
    "SRC-02": {
        "filename": "huetten_2024_deep_learning_visual_inspection.pdf",
        "title": "Deep Learning for Automated Visual Inspection in Manufacturing and Maintenance: A Survey of Open-Access Papers",
        "authors": "Nils Hütten; Miguel Alves Gomes; Florian Hölken; Karlo Andricevic; Richard Meyes; Tobias Meisen",
        "publication": "2024 (22 January 2024)",
        "publication_date": "2024-01-22",
        "doi": "https://doi.org/10.3390/asi7010011",
        "license": "CC BY 4.0; check third-party figure rights",
        "expected_pages": 38,
    },
}
source_paths = {}
print("--- Actual file listing in raw/ ---")
for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(RAW_DIR), "bytes:", p.stat().st_size)
for source_id, meta in SOURCES.items():
    path = RAW_DIR / meta["filename"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing {source_id}: {path}. Upload the two PDFs into the raw folder.")
    if path.stat().st_size < 1024:
        raise ValueError(f"Unusually small / possibly invalid PDF: {path}")
    source_paths[source_id] = path
print("INPUT VERIFICATION: PASS —", len(source_paths), "approved PDFs found")

In [ ]:
# A3 — Inspect both PDF files and actual extractable samples (required)
import importlib.util, subprocess, sys
if importlib.util.find_spec("fitz") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pymupdf"])
import fitz
inspection = {}
for sid, source in source_paths.items():
    with fitz.open(source) as pdf:
        page_counts = [len((page.get_text("text") or "").strip()) for page in pdf]
        first_samples = []
        for i in (0, min(1, len(pdf)-1)):
            sample = (pdf[i].get_text("text") or "").strip()
            first_samples.append({"page": i+1, "sample": " ".join(sample.split())[:220]})
        inspection[sid] = {
            "page_count": len(pdf),
            "extracted_characters": sum(page_counts),
            "empty_text_pages": sum(n == 0 for n in page_counts),
            "representative_samples": first_samples,
        }
        print("\n", sid, "|", source.name)
        print("Pages:", len(pdf), "| Extracted characters:", sum(page_counts), "| Empty pages:", inspection[sid]["empty_text_pages"])
        for item in first_samples:
            print(f"  Page {item['page']} example:", repr(item["sample"][:160]))
        assert len(pdf) == SOURCES[sid]["expected_pages"], "Unexpected page count; verify source version"
        assert sum(page_counts) > 1000, "Insufficient text; check scan/OCR need"
print("A3 INSPECTION: PASS — two genuine PDF sources inspected, not just file existence")

### A4 — Data inventory checkpoint

The corresponding submission-ready table is in `docs/week3_data_inventory.md`. The PDFs include diagrams, figure captions, technical tables and reference lists; extraction is limited to embedded text. Two-column reading order and source figures require manual review. These limitations are **observations about source layout**, not claims that the RAG system already works.

In [ ]:
# CELL 4 — REQUIRED: explicit selection and SHA-256 provenance helpers
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re

def select_source(relative_path):
    candidate = (RAW_DIR / relative_path).resolve()
    candidate.relative_to(RAW_DIR.resolve())  # disallow unrelated paths
    if not candidate.is_file():
        raise FileNotFoundError(f"Explicit source missing: {candidate}")
    return candidate

def source_hash(path):
    dig = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for buffer in iter(lambda: stream.read(1024*1024), b""):
            dig.update(buffer)
    return dig.hexdigest()

def save_manifest(records, derived=None):
    doc = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
        "derived_outputs": derived or [],
        "note": "Part B lineage: does not establish RAG accuracy, permission for third-party graphics, or source-text completeness.",
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.write_text(json.dumps(doc, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return doc

assert len([select_source(meta['filename']) for meta in SOURCES.values()]) == 2
print("CELL 4: source selection and manifest helpers ready")

In [ ]:
# CELL 4A — REQUIRED: common lineage registry, one source-to-output relation per PDF
lineage_entries = {}
def register_lineage(source, output, processing):
    source, output = Path(source).resolve(), Path(output).resolve()
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    if not source.is_file(): raise FileNotFoundError(source)
    if not output.is_file() or output.stat().st_size == 0: raise ValueError(f"Missing/empty prepared output: {output}")
    if not processing.strip(): raise ValueError("A nonempty processing method is required")
    lineage_entries[(str(source), str(output))] = {
        "source": source, "output": output, "processing": processing,
    }
    print("Registered:", source.name, "->", output.name)
print("CELL 4A: lineage registry initialized")

In [ ]:
# CELL 5A1 — CONDITIONAL REQUIRED Pathway A: inspect/prepare TWO exact source PDFs
# Purpose: compare fixed-size and paragraph-/block-aware text preparation.
# Uses PyMuPDF for explicit page labels; this is NOT semantic chunking or OCR.
import fitz
CHUNK_SIZE = 500
MIN_REVIEW_LENGTH = 60
CHOSEN_METHOD = "paragraph-aware"

def clean_block(raw):
    """Conservative text cleanup: keep technical content and mathematical symbols."""
    raw = raw.replace("\u00ad", "").replace("\xa0", " ")
    raw = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", raw)
    raw = re.sub(r"(?<=\w)-\n(?=[a-z])", "", raw)  # cautious line-wrap dehyphenation
    raw = re.sub(r"[ \t]+", " ", raw)
    return " ".join(line.strip() for line in raw.splitlines() if line.strip()).strip()

def inspect_pdf_blocks(path):
    """Extract PDF text blocks, retain exact page, skip only recognized page furniture.

    The recurring '1 3' and 'Vol.:(0123456789)' Springer margin markers are seen in SOURCE 01; their removal
    is an explicit rule, not a general ban on short text. Title-page metadata stays.
    """
    pages = []
    page_furniture_removed = 0
    with fitz.open(path) as pdf:
        for index, page in enumerate(pdf, start=1):
            blocks = []
            for b in page.get_text("blocks", sort=False):
                if len(b) < 7 or b[6] != 0:
                    continue  # non-text image block
                x0, y0, x1, y1, raw, _, _ = b
                cleaned = clean_block(raw)
                if not cleaned:
                    continue
                # The existing header/footer filter is intentionally conservative.
                if index > 1 and (y1 < 0.075 * page.rect.height or y0 > 0.925 * page.rect.height):
                    page_furniture_removed += 1
                    continue
                if path.name.startswith("ren_2022_") and (re.fullmatch(r"1\s+3", cleaned) or cleaned == "Vol.:(0123456789)"):
                    page_furniture_removed += 1
                    continue
                blocks.append(cleaned)
            if not blocks:
                raise ValueError(f"No usable PDF text blocks on page {index}: {path.name}; review possible OCR need")
            pages.append({"page": index, "blocks": blocks, "text": "\n\n".join(blocks)})
    return pages, page_furniture_removed

def split_around_words(s, cap):
    """Split text at whitespace; rebalance an orphan tail from the SAME block."""
    pieces = []
    remaining = s.strip()
    while len(remaining) > cap:
        split = remaining.rfind(" ", 1, cap + 1)
        if split < cap * 0.5:
            split = cap  # last-resort break for an unusually long token
        pieces.append(remaining[:split].strip())
        remaining = remaining[split:].strip()
    if remaining:
        pieces.append(remaining)
    if len(pieces) >= 2 and len(pieces[-1]) < MIN_REVIEW_LENGTH:
        tail = pieces.pop()
        prior = pieces.pop()
        joined = prior + " " + tail
        if len(joined) <= cap:
            pieces.append(joined)
        else:
            # Repartition at word boundary nearest the midpoint, both <= cap.
            middle = len(joined) // 2
            indices = [i for i, ch in enumerate(joined) if ch == " "
                       and 0 < i < len(joined)-1 and i <= cap
                       and len(joined)-i-1 <= cap]
            if indices:
                split = min(indices, key=lambda idx: abs(idx-middle))
                pieces.extend([joined[:split].strip(), joined[split+1:].strip()])
            else:
                pieces.extend([prior, tail])  # preserve hard-to-split technical token
    return [p for p in pieces if p]

def fixed_size_pages(pages, size=500):
    return [{"page": entry["page"], "text": entry["text"][pos:pos+size]}
            for entry in pages for pos in range(0, len(entry["text"]), size)]

def paragraph_aware_pages(pages, size=500):
    """Page/block-aware packing. Keep small meaningful headings, never mix PDF pages."""
    chunks=[]
    for entry in pages:
        current=""
        for block in entry["blocks"]:
            for segment in split_around_words(block,size):
                sep = "\n\n" if current else ""
                if current and len(current)+len(sep)+len(segment) > size:
                    chunks.append({"page": entry["page"], "text": current})
                    current = segment
                else:
                    current += sep + segment
        if current:
            chunks.append({"page": entry["page"], "text": current})
    return chunks

def fixed_midword_cuts(chunks):
    return sum(a["page"]==b["page"] and a["text"] and b["text"]
               and a["text"][-1].isalnum() and b["text"][0].isalnum()
               for a,b in zip(chunks,chunks[1:]))

def forced_midword_cuts(pages,size):
    count=0
    for entry in pages:
        for block in entry["blocks"]:
            s=block.strip()
            while len(s)>size:
                ix=s.rfind(" ",1,size+1)
                if ix < size*0.5: ix=size
                if ix == size and s[ix-1].isalnum() and s[ix].isalnum(): count+=1
                s=s[ix:].strip()
    return count

all_pages={}
comparison={}
for sid,meta in SOURCES.items():
    pages, dropped = inspect_pdf_blocks(select_source(meta["filename"]))
    all_pages[sid]=pages
    fixed=fixed_size_pages(pages,CHUNK_SIZE)
    paragraph=paragraph_aware_pages(pages,CHUNK_SIZE)
    comparison[sid]={
        "fixed_chunk_count":len(fixed),
        "paragraph_chunk_count":len(paragraph),
        "fixed_midword_boundaries":fixed_midword_cuts(fixed),
        "paragraph_forced_midword_boundaries":forced_midword_cuts(pages,CHUNK_SIZE),
        "paragraph_short_fragments_under_60":sum(len(x["text"]) < MIN_REVIEW_LENGTH for x in paragraph),
        "page_furniture_blocks_removed":dropped,
        "fixed_sample":fixed[0]["text"][:150],
        "paragraph_sample":paragraph[0]["text"][:150],
    }
    print(sid, "fixed:",len(fixed),"paragraph:",len(paragraph),
          "mid-word fixed/paragraph:",fixed_midword_cuts(fixed),"/",forced_midword_cuts(pages,CHUNK_SIZE),
          "short text fragments:",comparison[sid]["paragraph_short_fragments_under_60"],
          "recognized page-furniture blocks removed:",dropped)
    print("Fixed sample:",repr(fixed[0]["text"][:120]))
    print("Paragraph-aware sample:",repr(paragraph[0]["text"][:120]))
print("CELL 5A1: comparison done on both PDFs")

In [ ]:
# CELL 5A2 — REQUIRED: prepare traceable page-numbered JSONL records from BOTH PDFs
# Shared function will be reused in Part C with precisely the same configuration.
PREPARATION_CONFIG = {"method": CHOSEN_METHOD, "chunk_size": CHUNK_SIZE,
                      "source_ids": sorted(SOURCES.keys()), "page_metadata": True, "cleanup_version":"v1.1-known-footer-and-orphan-tail"}

def prepare_pdf_records(source_id, source_path, config):
    meta = SOURCES[source_id]
    pages, _ = inspect_pdf_blocks(source_path)
    method = config["method"]
    size = config["chunk_size"]
    if method == "paragraph-aware":
        chunks = paragraph_aware_pages(pages, size)
    elif method == "fixed":
        chunks = fixed_size_pages(pages, size)
    else:
        raise ValueError("Expected 'fixed' or 'paragraph-aware'")
    if not chunks: raise ValueError("No text chunks produced")
    checksum = source_hash(source_path)
    records = []
    for i, item in enumerate(chunks, start=1):
        records.append({
            "record_id": f"{source_id}-{i:05d}", "source_id": source_id,
            "source": meta["filename"], "source_title": meta["title"],
            "source_url": meta["doi"], "source_sha256": checksum,
            "publication": meta["publication"], "publication_date": meta["publication_date"],
            "access_rights": meta["license"], "page": item["page"],
            "processing": f"{method}-page-block-char-{size}",
            "text": item["text"],
        })
    return records

def write_jsonl(target, records):
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8") as handle:
        for row in records:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    if not target.is_file() or target.stat().st_size <= 0:
        raise ValueError("Prepared output missing or empty")
    return target

def prepare_all(target_dir, config):
    individual = {}
    combined = []
    for sid in config["source_ids"]:
        src = select_source(SOURCES[sid]["filename"])
        records = prepare_pdf_records(sid, src, config)
        target = write_jsonl(target_dir / (Path(src).stem + "_prepared_text.jsonl"), records)
        individual[sid] = target
        combined.extend(records)
    combined_path = write_jsonl(target_dir / "prepared_text.jsonl", combined)
    return individual, combined_path, combined

partb_individual, prepared_path, prepared_records = prepare_all(PROCESSED_DIR, PREPARATION_CONFIG)
assert all(row["page"] is not None and row["text"].strip() for row in prepared_records)
assert len({row["source_id"] for row in prepared_records}) == 2
print("CELL 5A2: prepared JSONL:", prepared_path)
print("Selected method:", CHOSEN_METHOD, "| Total chunks:", len(prepared_records))
print("Sample record:", {k: prepared_records[0][k] for k in ("record_id","source_id","page","source_url","processing")})

In [ ]:
# B3 — REQUIRED: quantitative comparison + basic JSONL content/stage validation
for sid, metrics in comparison.items():
    print(f"{sid}: fixed={metrics['fixed_chunk_count']} (midword={metrics['fixed_midword_boundaries']}); "
          f"paragraph-aware={metrics['paragraph_chunk_count']} "
          f"(forced midword={metrics['paragraph_forced_midword_boundaries']}, "
          f"short<60={metrics['paragraph_short_fragments_under_60']})")
    assert metrics['paragraph_forced_midword_boundaries'] <= metrics['fixed_midword_boundaries']
assert prepared_path.exists() and prepared_path.stat().st_size > 0, "Missing or empty Part B output"
loaded = [json.loads(line) for line in prepared_path.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(loaded) == len(prepared_records)
for row in loaded:
    assert row['text'].strip() and row['record_id'] and row['source_url'] and row['source_title']
    assert 1 <= row['page'] <= SOURCES[row['source_id']]['expected_pages']
    assert row['source_sha256'] == source_hash(select_source(row['source']))
    assert len(row['text']) <= CHUNK_SIZE
assert not any(row['source_id']=='SRC-01' and row['text'].strip()=='1 3' for row in loaded)
print("B3 OUTPUT VALIDATION: PASS —",len(loaded),"nonempty JSONL records")
print("First record preview:",loaded[0]['record_id'],loaded[0]['page'],repr(loaded[0]['text'][:170]))
print("Caveat: formatting, headings, equations, tables and diagrams still require manual review")

In [ ]:
# B4.1 — Register one Part B lineage relationship for EACH of the two PDFs
for sid in PREPARATION_CONFIG["source_ids"]:
    register_lineage(
        source=select_source(SOURCES[sid]["filename"]),
        output=partb_individual[sid],
        processing=f"{CHOSEN_METHOD} page/block-aware chunks, max {CHUNK_SIZE} chars; source title, DOI and page metadata retained"
    )
assert len(lineage_entries) == 2
print("B4.1: registered", len(lineage_entries), "source-specific relationships")

In [ ]:
# CELL 6A — REQUIRED: Part B common lineage manifest
lineage_records=[]
for i, entry in enumerate(lineage_entries.values(), start=1):
    source, output = entry["source"], entry["output"]
    sid = next(k for k,m in SOURCES.items() if m["filename"] == source.name)
    lineage_records.append({
        "source_id": sid,
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_title": SOURCES[sid]["title"],
        "source_url": SOURCES[sid]["doi"],
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": entry["processing"],
        "source_version": SOURCES[sid]["publication"],
        "access_rights": SOURCES[sid]["license"],
        "status": "Prepared; Topic 4 data quality review pending",
    })
manifest = save_manifest(lineage_records, derived=[{
    "prepared_output": str(prepared_path.relative_to(BASE_DIR)),
    "prepared_sha256": source_hash(prepared_path),
    "contains_source_ids": sorted(SOURCES),
    "processing_note": "Concatenation of source-specific page-labeled prepared chunks"
}])
print("CELL 6A: Part B lineage records:", len(manifest["records"]))

In [ ]:
# CELL 6B — REQUIRED: check saved Part B manifest against actual files and hashes
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert len(saved_manifest["records"]) == len(source_paths) == 2
for entry in saved_manifest["records"]:
    src = RAW_DIR / entry["source_filename"]
    out = BASE_DIR / entry["prepared_output"]
    assert src.is_file() and out.is_file()
    assert source_hash(src) == entry["source_sha256"]
    assert source_hash(out) == entry["prepared_sha256"]
for item in saved_manifest["derived_outputs"]:
    out = BASE_DIR / item["prepared_output"]
    assert out.is_file() and source_hash(out) == item["prepared_sha256"]
print("COMMON LINEAGE VALIDATION: PASS — 2 source-specific outputs + aggregate prepared JSONL")

### C1 — Integrated V1 pipeline

Re-run the **same** Part B transformation on the **same** two approved PDFs. Source hashes, prepared outputs and mapping must remain stable on an immediate repeat in the same environment; generated timestamps are expected to differ. This is a **repeatability smoke test**, not a proof of full cross-environment reproducibility.

In [ ]:
# CELL 7 — REQUIRED: pipeline utilities (the pipeline is batch ETL, not model inference)
PIPELINE_RUNS = {}

def execute_pipeline(kind, config):
    inputs = [select_source(SOURCES[sid]["filename"]) for sid in config["source_ids"]]
    inspections = {}
    for sid in config["source_ids"]:
        src = select_source(SOURCES[sid]["filename"])
        with fitz.open(src) as pdf:
            nonempty = sum(bool((page.get_text("text") or "").strip()) for page in pdf)
            inspections[sid] = {"pages": len(pdf), "pages_with_extractable_text": nonempty}
            if nonempty != len(pdf): raise ValueError(f"Blank-text page(s) in {src}")
    target_dir = PIPE_DIR / kind
    target_dir.mkdir(parents=True, exist_ok=True)
    per_source, combined_target, rows = prepare_all(target_dir, config)
    outputs = [per_source[sid] for sid in config["source_ids"]] + [combined_target]
    for item in outputs:
        if not item.is_file() or item.stat().st_size == 0: raise ValueError(f"Missing/empty output {item}")
    if len(rows) != sum(1 for _ in combined_target.open(encoding="utf-8")):
        raise ValueError("JSONL row count changed during export")
    mappings = [
        {"source": f"data/raw/{SOURCES[sid]['filename']}",
         "prepared_output": str(per_source[sid].relative_to(BASE_DIR)),
         "source_sha256": source_hash(select_source(SOURCES[sid]["filename"]))}
        for sid in config["source_ids"]
    ]
    validation = {
        "record_count": len(rows),
        "output_count": len(outputs),
        "output_sha256": {str(p.relative_to(BASE_DIR)): source_hash(p) for p in outputs},
        "output_bytes": {str(p.relative_to(BASE_DIR)): p.stat().st_size for p in outputs},
        "nonempty_page_references": all(isinstance(row["page"], int) and row["page"]>0 for row in rows),
    }
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": dict(config),
        "processing_method": f"batch ETL: PDF text blocks → cleaned page-labeled {config['method']} {config['chunk_size']}-char JSONL",
        "sources": [{"source_id": sid,
                     "path": f"data/raw/{SOURCES[sid]['filename']}",
                     "title": SOURCES[sid]["title"],
                     "doi": SOURCES[sid]["doi"],
                     "sha256": source_hash(select_source(SOURCES[sid]["filename"])),
                     "publication_date": SOURCES[sid]["publication_date"]}
                    for sid in config["source_ids"]],
        "outputs": [str(p.relative_to(BASE_DIR)) for p in outputs],
        "source_output_mapping": mappings,
        "combined_output": str(combined_target.relative_to(BASE_DIR)),
        "inspection": inspections,
        "validation": validation,
        "source_version": "Ren 2022; Hütten 2024",
        "access_rights": "CC BY 4.0 identified in both papers; check third-party content and attribution when redistributing",
        "limitations": "PDF multi-column, equations, tables and figures require later manual review; no embeddings or RAG evaluation performed",
    }
    target = target_dir / "pipeline_manifest.json"
    target.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Prepared outputs: {len(outputs)} | Chunks: {len(rows)}")
    print("Pipeline manifest:", target)
    return record

def register_and_run(kind, config):
    runner = lambda: execute_pipeline(kind, config)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first
print("CELL 7: pipeline utilities ready")

In [ ]:
# CELL 8A — CONDITIONAL REQUIRED: run ONLY selected PDF/text documents pipeline
run_A = register_and_run("documents", PREPARATION_CONFIG)
print("CELL 8A completed successfully. Extracted source PDFs are document knowledge, NOT image training samples.")

In [ ]:
# CELL 9 — REQUIRED: deterministic repeatability and provenance checks
assert PIPELINE_RUNS, "No registered pipeline"
repeatability = {"pipeline_version":"week3-v1", "generated_at_utc":datetime.now(timezone.utc).isoformat(),
                 "type":"same-environment smoke test", "checks":[]}
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], "Input fingerprints changed"
    assert first["validation"] == second["validation"], "Output hashes/counts/sizes changed"
    assert first["source_output_mapping"] == second["source_output_mapping"], "Source-output mapping changed"
    result = {"data_type":kind, "status":"PASS", "sources_equal":True,
              "validation_equal":True, "mappings_equal":True,
              "chunk_count":second["validation"]["record_count"]}
    repeatability["checks"].append(result)
    print("PASS:", kind, "— source hashes, output hashes/sizes and mappings are stable")
repeatability_path = PIPE_DIR / "documents" / "repeatability_evidence.json"
repeatability_path.write_text(json.dumps(repeatability, indent=2), encoding="utf-8")
print("All selected pipelines passed repeatability smoke checks.")
print("Evidence:", repeatability_path)

### C2–C4 — Outputs and lab submission

- `docs/week3_pipeline_diagram.png`: actual SOURCE → INGEST → INSPECT → PREPARE → VALIDATE → LINEAGE workflow plus two failure responses. Editable companion `.drawio` included.
- `docs/week3_pipeline_decisions.md`: configuration comparison, evidence, stage contracts, ETL/ELT and batch/streaming rationale, scope/rights limitations.
- `docs/week3_data_inventory.md`: two actual source PDFs, dates, rights, inspection samples and requirement mapping.
- `data/processed/prepared_text.jsonl`: Part B combined page-labelled records, with separate source-specific prepared outputs.
- `data/week3_manifest.json`: Part B provenance manifest.
- `data/pipeline_v1/documents/`: Part C prepared outputs, execution manifest and smoke-test evidence.

**DATA QUALITY NOTE:** Only recognized repeating page markers are omitted; short text is kept/merged where safe. The two PDFs cannot establish comprehensive AOI/AXI coverage.

**IMPORTANT:** Re-run in your own Colab runtime after uploading both PDFs, then download the outputs for GitHub if you need execution proof. This notebook has no API key or model dependencies. Future RAG answer evaluation is outside Topic 3.

In [ ]:
# OPTIONAL Cell 10 — Export Colab-generated data outputs for your existing GitHub repository.
# Run this cell with EXPORT_FOR_DOWNLOAD=True after the other steps are completed.
# Raw PDFs are intentionally EXCLUDED from this ZIP.
EXPORT_FOR_DOWNLOAD = False
if EXPORT_FOR_DOWNLOAD:
    from zipfile import ZipFile, ZIP_DEFLATED
    export_zip = BASE_DIR / "week3_colab_outputs.zip"
    with ZipFile(export_zip, "w", ZIP_DEFLATED) as archive:
        for folder in (PROCESSED_DIR, PIPE_DIR):
            for path in folder.rglob("*"):
                if path.is_file(): archive.write(path, path.relative_to(BASE_DIR))
        lineage_path = BASE_DIR / "data" / "week3_manifest.json"
        archive.write(lineage_path, lineage_path.relative_to(BASE_DIR))
    print("Export created:", export_zip)
    try:
        from google.colab import files
        files.download(str(export_zip))
    except ImportError:
        print("Not in Colab. Copy the exported ZIP directly from:", export_zip)
else:
    print("Optional export disabled. Set EXPORT_FOR_DOWNLOAD=True and rerun this cell if needed.")